In [ ]:
# # 🕵️ Analyse et Prédiction des Crimes à Los Angeles (LAPD)
# *Ce notebook présente la création d'un modèle de Deep Learning (MLP) pour prédire la probabilité de crimes spécifiques selon le profil de la victime et le lieu.*

# ## 1. Configuration et Chargement
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Montage Google Drive (si nécessaire)
# from google.colab import drive
# drive.mount('/content/drive')

# Chargement des données
path = '/content/drive/MyDrive/Analyzing-Crime-in-Los-Angeles-main/crimes.csv'
df = pd.read_csv(path)

# ## 2. Nettoyage et Préparation
# On filtre les âges aberrants
df = df[df['Vict Age'] > 0]

# Sélection des deux crimes cibles pour le Multi-Label
# Nous choisissons le Vol d'Identité et l'Agression Simple
df_id = df[df['Crm Cd Desc'] == 'THEFT OF IDENTITY']
df_agression = df[df['Crm Cd Desc'] == 'BATTERY - SIMPLE ASSAULT']

# Rééquilibrage du dataset (Undersampling des autres crimes pour éviter les biais)
df_calme = df[(df['Crm Cd Desc'] != 'THEFT OF IDENTITY') &
              (df['Crm Cd Desc'] != 'BATTERY - SIMPLE ASSAULT')].sample(len(df_id), random_state=42)

df_balanced = pd.concat([df_id, df_agression, df_calme]).sample(frac=1, random_state=42)

# ## 3. Ingénierie des Caractéristiques (Features)
X_raw = df_balanced[['AREA NAME', 'Vict Age', 'TIME OCC']]

# Encodage One-Hot des quartiers
X_encoded = pd.get_dummies(X_raw, columns=['AREA NAME'], drop_first=True)
model_columns = list(X_encoded.columns)

# Sauvegarde des colonnes pour l'application future
joblib.dump(model_columns, 'model_columns.pkl')

# Création des cibles (Y)
y1 = (df_balanced['Crm Cd Desc'] == 'THEFT OF IDENTITY').astype(int)
y2 = (df_balanced['Crm Cd Desc'] == 'BATTERY - SIMPLE ASSAULT').astype(int)
Y = np.column_stack((y1, y2))

# ## 4. Normalisation et Division
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
joblib.dump(scaler, 'scaler.pkl')

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, stratify=Y, random_state=42)

# ## 5. Construction du Modèle MLP (Multi-Layer Perceptron)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.2), # Évite le surapprentissage
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(2, activation='sigmoid') # 2 sorties indépendantes (Probabilités)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entraînement
history = model.fit(X_train, Y_train, epochs=25, batch_size=32, validation_split=0.2, verbose=1)

# Sauvegarde du modèle
model.save('mon_modele_mlp.h5')

# ## 6. Évaluation des Performances
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Perte Entraînement')
plt.plot(history.history['val_loss'], label='Perte Validation')
plt.title('Évolution de la Perte')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Précision Entraînement')
plt.plot(history.history['val_accuracy'], label='Précision Validation')
plt.title('Évolution de la Précision')
plt.legend()
plt.show()

# Rapport de classification
y_pred = (model.predict(X_test) > 0.5).astype(int)
print("\n🔥 RAPPORT FINAL - VOL D'IDENTITÉ :\n", classification_report(Y_test[:, 0], y_pred[:, 0]))
print("\n🔥 RAPPORT FINAL - AGRESSION :\n", classification_report(Y_test[:, 1], y_pred[:, 1]))